In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import polars as pl

# --- CONFIG ---
emb_model = "qwen3-8b-with-instruction"
path = f'../../data/raw/{emb_model}_embeddings_raw.parquet'
n_dims = 1024

# Read in Data
item_embeddings = pl.read_parquet(path)

# Get the number of embeddings in item_embeddings
n_embeddings = len([col for col in item_embeddings.columns if col.startswith("emb")])

# Use GPU if available; for model-train speed
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Convert embeddings columns to a torch tensor
X_train = item_embeddings.select(pl.col('^emb.*$')).to_numpy()
X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

# Define Architecture
class PsychometricAE(nn.Module):
    def __init__(self, input_dim=n_embeddings, bottleneck_dim=n_dims):
        super().__init__()
        # Encoder: Funneling down to the signal
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, bottleneck_dim) # Linear bottleneck to preserve distance logic
        )
        # Decoder: Reconstructing to ensure information was kept
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Training Setup
model = PsychometricAE().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop 
print("Training Autoencoder...")
for epoch in range(500):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, X_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/500], Loss: {loss.item():.6f}")

# Extract Compressed Features
with torch.no_grad():
    compressed_vectors = model.encoder(X_tensor).cpu().numpy()

column_names = [f"emb_{i+1}" for i in range(n_dims)]

# Return to Polars
embeddings_compressed = (
    item_embeddings.with_columns(
        pl.Series("compressed_embedding", [list(v) for v in compressed_vectors])
    )
    .select([
        pl.col("item"),
        pl.col("compressed_embedding").list.to_struct(
            fields=column_names # Names the columns
        )
    ])
    .unnest("compressed_embedding")
)

print("Success! Compressed features added to dataframe.")

Training Autoencoder...
Epoch [20/500], Loss: 0.000126
Epoch [40/500], Loss: 0.000119
Epoch [60/500], Loss: 0.000105
Epoch [80/500], Loss: 0.000093
Epoch [100/500], Loss: 0.000081
Epoch [120/500], Loss: 0.000074
Epoch [140/500], Loss: 0.000068
Epoch [160/500], Loss: 0.000065
Epoch [180/500], Loss: 0.000060
Epoch [200/500], Loss: 0.000058
Epoch [220/500], Loss: 0.000053
Epoch [240/500], Loss: 0.000051
Epoch [260/500], Loss: 0.000049
Epoch [280/500], Loss: 0.000048
Epoch [300/500], Loss: 0.000045
Epoch [320/500], Loss: 0.000045
Epoch [340/500], Loss: 0.000042
Epoch [360/500], Loss: 0.000041
Epoch [380/500], Loss: 0.000039
Epoch [400/500], Loss: 0.000038
Epoch [420/500], Loss: 0.000037
Epoch [440/500], Loss: 0.000038
Epoch [460/500], Loss: 0.000035
Epoch [480/500], Loss: 0.000036
Epoch [500/500], Loss: 0.000033
Success! Compressed features added to dataframe.


In [52]:
# Save the weights to disk
torch.save(model.state_dict(), '../../models/psychometric_ae_weights.pt')
print("Model saved successfully!")


Model saved successfully!


In [53]:
import os

# Read in item correlations data and item list with transformers features
item_cors_df = pl.read_parquet(f"../../data/processed/item_correlations_with_cross.parquet")
item_list_df = pl.read_parquet(f"../../data/processed/item_list_sentiment.parquet")

# Join compressed embeddings into item_list_df to combine features
items_embedded = item_list_df.join(
    embeddings_compressed, 
    on='item',
    how='left'
)

# Create Subfolders if they do not exist yet
os.makedirs(f"../../data/clustered_embeddings", exist_ok=True)
os.makedirs(f"../../data/clustered_embeddings/{emb_model}", exist_ok=True)

# Save Item list with all features to disk
items_embedded.write_parquet(f"../../data/clustered_embeddings/{emb_model}/item_list_emb_{n_dims}.parquet")

In [54]:
# Join embeddings per item into the pairwise item correlations dataframe
item_cors_df = item_cors_df.join(
    embeddings_compressed, right_on='item', left_on='Parameter1', how='left', suffix='_item1'
).join(
    embeddings_compressed, right_on='item', left_on='Parameter2', how='left', suffix='_item2'
)

# Exclude mean, sd, min, max from the item_list_df to prevent leaks
item_list_df = item_list_df.select(
        pl.exclude(['mean', 'sd', 'max', 'min', 'item_right'])
)


In [55]:
import polars.selectors as cs
import numpy as np
from numpy.linalg import norm

# Define arrays of embeddings columns per item
cols_1 = [f"emb_{i+1}" for i in range(n_dims)]
cols_2 = [f"emb_{i+1}_item2" for i in range(n_dims)]

# 1. Product Features (Element-wise)
# Basically cosine similarity using Point estimates instead of vectors
product_features = [
    (pl.col(c1) * pl.col(c2)).alias(f"prod_{i+1}") 
    for i, (c1, c2) in enumerate(zip(cols_1, cols_2))
]

# 2. Global Cosine Similarity 
# Cosine = Dot Product / (Norm A * Norm B)
dot_product = pl.sum_horizontal([pl.col(c1) * pl.col(c2) for c1, c2 in zip(cols_1, cols_2)])
norm_a = pl.sum_horizontal([pl.col(c)**2 for c in cols_1]).sqrt()
norm_b = pl.sum_horizontal([pl.col(c)**2 for c in cols_2]).sqrt()

# Add in global similarity feature and exclude the raw item embeddings from final df
df_final = item_cors_df.with_columns(product_features).with_columns(
    global_sim = dot_product / (norm_a * norm_b)
).select(
    ~cs.starts_with("emb")
)

# Rename columns in item_list to have the suffix "_item1" or "_item2"
item_list_v1 = item_list_df.rename(lambda column_name: column_name + "_item1")
item_list_v2 = item_list_df.rename(lambda column_name: column_name + "_item2")

# Join in these item_list versions into final df
# Good for readability of feature importances while modelling
df_final = df_final.join(
    item_list_v1, 
    left_on='Parameter1',
    right_on='item_item1',
    how='left',
).join(
    item_list_v2,
    left_on='Parameter2',
    right_on='item_item2',
    how='left',
)

In [ ]:
# Overview of the dataframe
df_final.head()

Parameter1,Parameter2,r,pair_negative,pair_positive,contradiction,entail,logic_neutral,similarity,thematic_intensity,logical_friction,sentiment_balance,prod_1,prod_2,prod_3,prod_4,prod_5,prod_6,prod_7,prod_8,prod_9,prod_10,prod_11,prod_12,prod_13,prod_14,prod_15,prod_16,prod_17,prod_18,prod_19,prod_20,prod_21,prod_22,prod_23,prod_24,prod_25,…,prod_1013,prod_1014,prod_1015,prod_1016,prod_1017,prod_1018,prod_1019,prod_1020,prod_1021,prod_1022,prod_1023,prod_1024,global_sim,sent_positive_item1,sent_neutral_item1,sent_negative_item1,emo_neutral_item1,emo_surprise_item1,emo_joy_item1,emo_fear_item1,emo_anger_item1,emo_sadness_item1,emo_disgust_item1,top_sentiment_item1,top_emotion_item1,sent_positive_item2,sent_neutral_item2,sent_negative_item2,emo_neutral_item2,emo_surprise_item2,emo_joy_item2,emo_fear_item2,emo_anger_item2,emo_sadness_item2,emo_disgust_item2,top_sentiment_item2,top_emotion_item2
str,str,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""Do you often long for exciteme…","""Do you often need understandin…",0.201111,0.011116,0.678711,0.999023,0.000023,0.001013,0.016022,3.6862e-7,0.016006,0.667595,-0.00014,0.000261,0.000057,0.000266,0.000124,0.000039,-0.000192,0.000035,0.000283,0.000038,0.000073,-0.000284,0.00011,0.000126,-0.00001,-0.000087,0.000042,-0.000171,0.000507,-0.000096,0.000054,0.000946,0.000556,-0.000033,0.000364,…,0.000084,0.000125,0.001555,-0.000713,-0.000036,0.000201,-0.000405,0.000316,0.000037,-0.000006,0.000647,0.000937,0.398919,0.783085,0.20548,0.011435,0.784396,0.119588,0.032255,0.030119,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.597131,0.379189,0.023679,0.749233,0.072289,0.127554,0.005687,0.017466,0.013736,0.014034,"""positive""","""neutral"""
"""Do you often long for exciteme…","""Do you stop and think things o…",-0.013197,0.025635,0.254395,1.0,0.000011,0.000136,0.006691,7.2584e-8,0.006691,0.22876,0.000248,-0.000035,-0.000168,0.000577,0.000022,0.000047,0.000573,-0.000068,0.000273,0.00014,-0.000145,0.000347,-0.000084,0.000116,-0.000012,0.000107,0.000023,-0.000202,0.000122,-0.000726,-0.000134,0.000042,0.001347,-0.000072,0.000327,…,0.00007,0.000221,0.000639,-0.000272,-0.000106,0.000098,0.000797,0.000681,-0.000063,9.5470e-7,-0.000106,0.000599,0.261563,0.783085,0.20548,0.011435,0.784396,0.119588,0.032255,0.030119,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.021467,0.800829,0.177705,0.062313,0.012318,0.002852,0.072816,0.784762,0.01101,0.053928,"""neutral""","""anger"""
"""Do you often long for exciteme…","""If you say you will do somethi…",-0.025196,0.038971,0.341797,0.011948,0.000206,0.987793,0.002369,4.8742e-7,0.000028,0.302826,0.000225,0.000371,0.000013,0.000541,0.000013,0.000035,0.000464,0.000041,0.000207,0.000104,-0.000329,0.000018,-0.000002,0.000061,0.000002,-0.000284,0.000074,-0.000136,0.000121,-0.000681,-0.000084,0.000918,0.000861,0.000109,-0.000066,…,0.000052,-0.000104,0.001011,0.000017,-0.00019,0.000158,0.000569,0.000568,-0.000067,0.000006,0.00067,0.000785,0.245872,0.783085,0.20548,0.011435,0.784396,0.119588,0.032255,0.030119,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.053004,0.696144,0.250853,0.51845,0.006533,0.004173,0.020429,0.382281,0.01385,0.054285,"""neutral""","""neutral"""
"""Do you often long for exciteme…","""Do your moods go up and down?""",0.135682,0.02153,0.2771,0.974609,0.001226,0.024017,0.02037,0.000025,0.019853,0.255569,0.000078,0.000179,0.000032,-0.000079,-0.000144,0.00019,0.000099,-0.00003,-0.000139,-0.000018,0.000134,0.000507,0.000007,0.000017,0.00002,-0.000076,0.000007,-0.000098,0.000761,0.000352,0.000055,0.001555,0.000579,0.000042,0.000182,…,-0.000042,-0.000258,0.002022,-0.000389,-0.000079,0.000294,0.000464,-0.000134,-0.000054,0.000005,0.000099,0.000166,0.527315,0.783085,0.20548,0.011435,0.784396,0.11

In [ ]:
# Save to disk
df_final.write_parquet(f"../../data/clustered_embeddings/{emb_model}/autoencoded_clusters_{n_dims}.parquet")